Step1: Importing Required libraries

In [1]:
import pandas as pd
import numpy as np

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

Step2: Loading the dataset

In [2]:

df = pd.read_csv("Segment.csv")

# Display the first five rows
print(df.head())

# Display column names
print(df.columns)

     ID  Year_Birth   Education Marital_Status   Income  Kidhome  Teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293.0        1         0   

  Dt_Customer  Recency  MntWines  ...  NumWebVisitsMonth  AcceptedCmp3  \
0  04-09-2012       58       635  ...                  7             0   
1  08-03-2014       38        11  ...                  5             0   
2  21-08-2013       26       426  ...                  4             0   
3  10-02-2014       26        11  ...                  6             0   
4  19-01-2014       94       173  ...                  5             0   

   AcceptedCmp4  AcceptedCmp5  AcceptedCmp1  AcceptedCmp2  Complain  \
0             0

step3: Data processing

In [3]:
# Create a copy of the dataset
df_processed = df.copy()

# Convert customer enrollment date to datetime format
df_processed["Dt_Customer"] = pd.to_datetime(
    df_processed["Dt_Customer"],
    dayfirst=True
)

# Create new features from the date
df_processed["Customer_Year"] = df_processed["Dt_Customer"].dt.year
df_processed["Customer_Month"] = df_processed["Dt_Customer"].dt.month
df_processed["Customer_Day"] = df_processed["Dt_Customer"].dt.day

Step4: Encode categorical variables

In [4]:
# so convert text values into numbers.

categorical_columns = [
    "Education",
    "Marital_Status"
]

encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()

    df_processed[column] = encoder.fit_transform(df_processed[column])

    encoders[column] = encoder

    print(f"\nEncoding for {column}")
    print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))



Encoding for Education
{'2n Cycle': np.int64(0), 'Basic': np.int64(1), 'Graduation': np.int64(2), 'Master': np.int64(3), 'PhD': np.int64(4)}

Encoding for Marital_Status
{'Absurd': np.int64(0), 'Alone': np.int64(1), 'Divorced': np.int64(2), 'Married': np.int64(3), 'Single': np.int64(4), 'Together': np.int64(5), 'Widow': np.int64(6), 'YOLO': np.int64(7)}


Step5: Dropping uncessary columns

In [5]:
# ID does not help prediction.
# Dt_Customer has already been converted into
# Year, Month and Day.

df_processed.drop(
    columns=["ID", "Dt_Customer"],
    inplace=True
)

print("\nProcessed Dataset")
print(df_processed.head())


Processed Dataset
   Year_Birth  Education  Marital_Status   Income  Kidhome  Teenhome  Recency  \
0        1957          2               4  58138.0        0         0       58   
1        1954          2               4  46344.0        1         1       38   
2        1965          2               5  71613.0        0         0       26   
3        1984          2               5  26646.0        1         0       26   
4        1981          4               3  58293.0        1         0       94   

   MntWines  MntFruits  MntMeatProducts  ...  AcceptedCmp5  AcceptedCmp1  \
0       635         88              546  ...             0             0   
1        11          1                6  ...             0             0   
2       426         49              127  ...             0             0   
3        11          4               20  ...             0             0   
4       173         43              118  ...             0             0   

   AcceptedCmp2  Complain  Z_CostCont

Step6: Defining features and target

In [6]:
# Features (Independent Variables)
X = df_processed.drop("Response", axis=1)

# Target Variable (Dependent Variable)
y = df_processed["Response"]


Step7: Splitting dataset

In [7]:
# 80% Training
# 20% Testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Training Samples: 1792
Testing Samples: 448


Step8: Creating classification models

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

Solving missing values in income

In [10]:
print(df_processed.isnull().sum())

Year_Birth              0
Education               0
Marital_Status          0
Income                 24
Kidhome                 0
Teenhome                0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Complain                0
Z_CostContact           0
Z_Revenue               0
Response                0
Customer_Year           0
Customer_Month          0
Customer_Day            0
dtype: int64


In [11]:
df_processed["Income"].fillna(df_processed["Income"].median(), inplace=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_17844\1234219816.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_processed["Income"].fillna(df_processed["Income"].median(), inplace=True)


0       58138.0
1       46344.0
2       71613.0
3       26646.0
4       58293.0
         ...   
2235    61223.0
2236    64014.0
2237    56981.0
2238    69245.0
2239    52869.0
Name: Income, Length: 2240, dtype: float64

In [12]:
print(df_processed.isnull().sum())

Year_Birth              0
Education               0
Marital_Status          0
Income                 24
Kidhome                 0
Teenhome                0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Complain                0
Z_CostContact           0
Z_Revenue               0
Response                0
Customer_Year           0
Customer_Month          0
Customer_Day            0
dtype: int64


In [13]:
print(df_processed["Income"].dtype)

float64


In [14]:
df_processed["Income"] = df_processed["Income"].fillna(
    df_processed["Income"].median()
)

In [15]:
print(df_processed["Income"].isnull().sum())

0


In [16]:
# Recreate X and y
X = df_processed.drop("Response", axis=1)
y = df_processed["Response"]

# Split again
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Step9: Training and evaluating models

In [17]:
for name, model in models.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Evaluate performance
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    cm = confusion_matrix(y_test, predictions)

    # Display results
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

    print("\nConfusion Matrix")
    print(cm)


Logistic Regression


c:\Users\DELL\Desktop\my documents\Recess2\KayuzaWilson_2400705616\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy : 0.8527
Precision: 0.5714
Recall   : 0.1739
F1 Score : 0.2667

Confusion Matrix
[[370   9]
 [ 57  12]]

Decision Tree
Accuracy : 0.8036
Precision: 0.3662
Recall   : 0.3768
F1 Score : 0.3714

Confusion Matrix
[[334  45]
 [ 43  26]]

Random Forest
Accuracy : 0.8638
Precision: 0.6429
Recall   : 0.2609
F1 Score : 0.3711

Confusion Matrix
[[369  10]
 [ 51  18]]


Step10: Predict a new Customer